# España: integración y cobertura

123 series integradas; cada una conserva su periodo, unidad y procedencia. Consulta `../output/revision_espana/INTEGRACION_FINAL.md` para límites y reproducción. Este notebook complementa el contraste AMECO del notebook 02; no certifica equivalencia numérica con MOISEES.

In [1]:
from pathlib import Path
import sys
import polars as pl
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
R=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(R/'src'))
from verify_spain_recovered import verify
verify()
a=pl.read_parquet(R/'datos/silver/espana.parquet')
c=pl.read_parquet(R/'datos/silver/espana_catalogo.parquet')
r=pl.read_parquet(R/'datos/silver/espana_fuentes_recuperadas.parquet')
display(c.select('series_id','name','unit','first','last','n','status'))

{'status': 'OK', 'series': 123, 'observations': 6434, 'new_series': 26, 'new_observations': 835, 'subannual_observations': 2544, 'independent_aggregation_checks': 422, 'partial_2004_excluded_from_analytic_view': True}


series_id,name,unit,first,last,n,status
str,str,str,i64,i64,i64,str
"""BDE.HIST.BM.YEAR_END""","""BM histórico España, cierre an…","""million_pesetas""",1962,1997,36,"""ACEPTADA_FUENTE_OFICIAL"""
"""BDE.HIST.M3.YEAR_END""","""M3 histórico España, cierre an…","""million_pesetas""",1962,1997,36,"""ACEPTADA_FUENTE_OFICIAL"""
"""BDE.HIST.M3_BM.YEAR_END""","""Multiplicador M3/BM, variante …","""ratio""",1962,1997,36,"""ACEPTADA_FUENTE_OFICIAL"""
"""ESP.1.0.0.0.AKGDV""","""Marginal efficiency of capital…","""Marginal efficiency of capital""",1961,2027,67,"""ACEPTADA_REVISION_MANUAL"""
"""ESP.1.0.0.0.AKNDV""","""Net capital stock per unit of …","""capital output ratio""",1960,2027,68,"""ACEPTADA_REVISION_MANUAL"""
…,…,…,…,…,…,…
"""PWT11.AVH""","""Horas anuales medias por perso…","""hours_per_person_year""",1950,2023,74,"""ACEPTADA_CONCEPTO_Y_FUENTE"""
"""PWT11.EMP""","""Personas ocupadas (PWT 11)""","""million_persons""",1950,2023,74,"""ACEPTADA_CONCEPTO_Y_FUENTE"""
"""SEPE.REGISTERED.UNEMPLOYED.ANN…","""Paro registrado, media de doce…","""persons""",2001,2025,25,"""ACEPTADA_FUENTE_OFICIAL"""


## Cobertura de las 26 series nuevas

In [2]:
coverage=r.group_by('series_id').agg(pl.col('year').min().alias('first'),pl.col('year').max().alias('last'),pl.len().alias('n')).sort('series_id')
display(coverage)
fig,ax=plt.subplots(figsize=(11,9))
for i,row in enumerate(coverage.iter_rows(named=True)):
    ax.plot([row['first'],row['last']],[i,i],linewidth=4)
ax.set_yticks(range(coverage.height),coverage['series_id'].to_list(),fontsize=8)
ax.set_xlabel('Año; el rango no implica igualdad de universo entre series')
ax.grid(axis='x',alpha=.3)
fig.tight_layout()
display(fig)
plt.close(fig)

series_id,first,last,n
str,i64,i64,u32
"""BDE.HIST.BM.YEAR_END""",1962,1997,36
"""BDE.HIST.M3.YEAR_END""",1962,1997,36
"""BDE.HIST.M3_BM.YEAR_END""",1962,1997,36
"""IMF.IFS.M1.ANNUAL_MEAN""",1962,1998,37
"""IMF.IFS.M1.YEAR_END""",1962,1998,37
…,…,…,…
"""MITES.SMI.MONTHLY_RATE.ANNUAL_…",1981,2025,45
"""MITES.SMI.MONTHLY_RATE.YEAR_EN…",1980,2025,46
"""SEPE.REGISTERED.UNEMPLOYED.ANN…",2001,2025,25


<Figure size 1100x900 with 1 Axes>

## Explorar una serie

Se muestran también los indicadores de dato parcial, la transformación y la fuente por año. Para explorar interactivamente, ejecuta las celdas con un kernel local.

In [3]:
def inspect(series):
    d=a.filter(pl.col('series_id')==series).sort('year')
    display(d.select('year','value','unit','source','apta_analisis','source_flag','transformation'))
    fig,ax=plt.subplots(figsize=(10,3))
    ax.plot(d['year'],d['value'],marker='.',linewidth=1)
    ax.set_title(d['name'][0]);ax.set_ylabel(d['unit'][0]);ax.grid(alpha=.3)
    display(fig);plt.close(fig)
selector=widgets.Dropdown(options=sorted(r['series_id'].unique().to_list()),description='Serie:',layout=widgets.Layout(width='95%'))
widgets.interact(inspect,series=selector)
inspect('SEPE.REGISTERED.UNEMPLOYED.ANNUAL_MEAN')

year,value,unit,source,apta_analisis,source_flag,transformation
i64,f64,str,str,bool,str,str
1962,132439.0,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""
1963,156358.0,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""
1964,183461.0,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""
1965,183461.0,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""
1966,229710.0,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""
…,…,…,…,…,…,…
1993,7.7914e6,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""
1994,8.5939e6,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""
1995,8.929e6,"""million_pesetas""","""bde_money_statistical_annex_20…",true,null,"""year_end_original"""


<Figure size 1000x300 with 1 Axes>

interactive(children=(Dropdown(description='Serie:', layout=Layout(width='95%'), options=('BDE.HIST.BM.YEAR_EN…

year,value,unit,source,apta_analisis,source_flag,transformation
i64,f64,str,str,bool,str,str
2001,1.9302e6,"""persons""","""sepe_registered_2001_2016_2026…",true,null,"""mean_twelve_months"""
2002,2.0496e6,"""persons""","""sepe_registered_2001_2016_2026…",true,null,"""mean_twelve_months"""
2003,2.0969e6,"""persons""","""sepe_registered_2001_2016_2026…",true,null,"""mean_twelve_months"""
2004,2.1137e6,"""persons""","""sepe_registered_2001_2016_2026…",true,null,"""mean_twelve_months"""
2005,2.0699e6,"""persons""","""sepe_registered_2001_2016_2026…",true,null,"""mean_twelve_months"""
…,…,…,…,…,…,…
2021,3565239.5,"""persons""","""sepe_registered_2017_current_2…",true,null,"""mean_twelve_months"""
2022,2.9628e6,"""persons""","""sepe_registered_2017_current_2…",true,null,"""mean_twelve_months"""
2023,2.7669e6,"""persons""","""sepe_registered_2017_current_2…",true,null,"""mean_twelve_months"""


<Figure size 1000x300 with 1 Axes>

## Periodos incompletos y anomalías de los originales

In [4]:
display(pl.read_csv(R/'output/revision_espana/periodos_completos_anualizacion.csv').filter(~pl.col('included')))
display(pl.read_csv(R/'output/revision_espana/source_anomalies.csv'))
display(r.filter(pl.col('source_flag').is_not_null()).select('series_id','year','value','source_flag','apta_analisis'))

series_id,year,n_periods,required,included
str,i64,i64,i64,bool
"""INE.HIST.SUSPENSIONS.TOTAL""",2004,3,4,false
"""INE.HIST.SUSPENSIONS.ASSETS""",2004,3,4,false
"""INE.HIST.SUSPENSIONS.LIABILITI…",2004,3,4,false
"""INE.HIST.BANKRUPTCIES.TOTAL""",2004,3,4,false
"""INE.HIST.BANKRUPTCIES.ASSETS""",2004,3,4,false
"""INE.HIST.BANKRUPTCIES.LIABILIT…",2004,3,4,false
"""SEPE.REGISTERED.UNEMPLOYED.ANN…",2026,9,12,false


source,row,issue,raw_token,year_block,selected_period,reason
str,i64,str,f64,i64,str,str
"""sepe_registered_2001_2016_2026…",97,"""INCONSISTENT_YEAR_FIELDS""",200801.0,2007,"""2008-01""","""Secuencia enero-diciembre cont…"
"""sepe_registered_2001_2016_2026…",98,"""INCONSISTENT_YEAR_FIELDS""",200802.0,2007,"""2008-02""","""Secuencia enero-diciembre cont…"
"""sepe_registered_2001_2016_2026…",99,"""INCONSISTENT_YEAR_FIELDS""",200803.0,2007,"""2008-03""","""Secuencia enero-diciembre cont…"
"""sepe_registered_2001_2016_2026…",100,"""INCONSISTENT_YEAR_FIELDS""",200804.0,2007,"""2008-04""","""Secuencia enero-diciembre cont…"
"""sepe_registered_2001_2016_2026…",101,"""INCONSISTENT_YEAR_FIELDS""",200805.0,2007,"""2008-05""","""Secuencia enero-diciembre cont…"
…,…,…,…,…,…,…
"""sepe_registered_2001_2016_2026…",156,"""INCONSISTENT_YEAR_FIELDS""",201108.0,2012,"""2012-08""","""Secuencia enero-diciembre cont…"
"""sepe_registered_2001_2016_2026…",157,"""INCONSISTENT_YEAR_FIELDS""",201109.0,2012,"""2012-09""","""Secuencia enero-diciembre cont…"
"""sepe_registered_2001_2016_2026…",158,"""INCONSISTENT_YEAR_FIELDS""",201110.0,2012,"""2012-10""","""Secuencia enero-diciembre cont…"


series_id,year,value,source_flag,apta_analisis
str,i64,f64,str,bool
"""INE.INSOLVENCY.ENTERPRISES""",2004,193.0,"""PARTIAL_YEAR_LAST_FOUR_MONTHS""",false
